# 06 — RNN profunda para producción diaria

(En revisión: experimento inicial, no sustituye el modelo semanal.) Usa una red recurrente de dos capas con cuatro semanas recientes por serie y rezagos estacionales del año anterior. El test es temporal; se conserva una máscara para diferenciar día sin registro de cero reportado. El clima sólo entra como historia conocida antes del origen cuando la cobertura alcanza 80%; si falta o está atrasado, la misma arquitectura corre sin variables climáticas. Las semanas incompletas no se puntúan.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"Datos_analiticos_proyecto").is_dir())
DATA=ROOT/"Datos_analiticos_proyecto"; OUT=DATA/"experimentos_actualizacion_20261020"; OUT.mkdir(exist_ok=True)
try:
    import torch
    from torch import nn
except ImportError as exc:
    raise ImportError("Este notebook requiere PyTorch CPU para entrenar la RNN profunda; instalar en entorno_tesis.") from exc
torch.manual_seed(42); np.random.seed(42)


In [2]:
# Construcción diaria: hueco sin registro = máscara 0, nunca se fuerza a cero real.
K=["finca","producto","color","variedad"]
old=pd.read_parquet(DATA/"produccion_real_limpio.parquet")
new=pd.read_parquet(DATA/"produccion_real_2026_actualizada.parquet")
old=old.loc[old.finca.eq("GAITANA")&old.llave_completa&old.fecha_corte.lt("2026-01-01")]
new=new.loc[new.finca.eq("GAITANA")&new.llave_completa&new.fecha_corte.ge("2026-01-01")]
d=pd.concat([old,new],ignore_index=True)
d["fecha"]=pd.to_datetime(d.fecha_corte).dt.normalize()
daily=d.groupby(K+["fecha"],as_index=False).agg(y=("tallos","sum"))
date_max=d.fecha.max();week_start=date_max-pd.Timedelta(days=date_max.weekday())
last_complete=week_start if date_max.weekday()==6 else week_start-pd.Timedelta(days=7)
print("Último día observado:",date_max.date(),"| última semana completa:",last_complete.date())
print("La RNN usa 28 días de secuencia reciente y el valor/estado de hace 364 días.")
cl=pd.read_parquet(DATA/"clima_limpio.parquet")
cl=cl.loc[cl.finca.eq("GAITANA")&cl.medicion_valida].copy()
cl["fecha"]=pd.to_datetime(cl.fecha).dt.normalize()
cl=cl.groupby("fecha",as_index=False).agg(temperatura=("temperatura","mean"),humedad=("humedad","mean"),radiacion=("radiacion","mean"))
climate_map=cl.set_index("fecha")
start=pd.Timestamp("2024-01-01"); end=last_complete+pd.Timedelta(days=6)
train_end=pd.Timestamp("2026-07-01"); test_start=pd.Timestamp("2026-09-01")
features=[];targets=[];scales=[];metas=[];year_values=[];cover=[]
series=daily.groupby(K,sort=False)
for key,g in series:
    s=g.groupby("fecha").y.sum().reindex(pd.date_range(start,end,freq="D"))
    observed=s.notna().to_numpy(); values=s.fillna(0).to_numpy(dtype=np.float32)
    dates=s.index
    scale=float(np.median(values[(dates<train_end)&observed&(values>0)])) if np.any((dates<train_end)&observed&(values>0)) else 0.
    if scale<=0 or int(observed.sum())<180: continue
    lag=np.r_[np.zeros(364,dtype=np.float32),values[:-364]]
    lagmask=np.r_[np.zeros(364,dtype=np.float32),observed[:-364].astype(np.float32)]
    dow=dates.dayofweek.to_numpy(); doy=dates.dayofyear.to_numpy()
    base=np.column_stack([np.log1p(values/scale),observed.astype(np.float32),np.log1p(lag/scale),lagmask,
        np.sin(2*np.pi*dow/7),np.cos(2*np.pi*dow/7),np.sin(2*np.pi*doy/365.25),np.cos(2*np.pi*doy/365.25)])
    wx=climate_map.reindex(dates)[["temperatura","humedad","radiacion"]].to_numpy(dtype=np.float32)
    wxmask=np.isfinite(wx).astype(np.float32); wx=np.nan_to_num(wx,nan=0.0)
    full=np.column_stack([base,wx,wxmask])
    for j in range(28,len(dates)):
        dt=dates[j]
        if not observed[j]: continue
        features.append(full[j-28:j]); targets.append(np.log1p(values[j]/scale)); scales.append(scale)
        metas.append((key,dt,values[j],values[j-1] if observed[j-1] else np.nan,
                      np.nanmean(values[max(0,j-7):j][observed[max(0,j-7):j]]) if observed[max(0,j-7):j].any() else np.nan,
                      values[j-364] if j>=364 and observed[j-364] else np.nan))
        year_values.append(j)
        cover.append(wxmask[j-28:j].mean())
X=np.asarray(features,dtype=np.float32); yt=np.asarray(targets,dtype=np.float32); sc=np.asarray(scales,dtype=np.float32)
meta=pd.DataFrame(metas,columns=["serie","fecha","y","ultimo_valor","media7_observada","mismo_dia_anio_anterior"])
train_mask=meta.fecha.lt(pd.Timestamp("2026-07-01"))&meta.fecha.ge(pd.Timestamp("2025-01-01"))
val_mask=meta.fecha.ge(pd.Timestamp("2026-07-01"))&meta.fecha.lt(test_start)
test_mask=meta.fecha.ge(test_start)&meta.fecha.le(end)
cov_train=float(np.mean(np.asarray(cover)[train_mask.to_numpy()]))
latest_test_date=meta.loc[test_mask,"fecha"].max()
recent_test=meta.fecha.ge(latest_test_date-pd.Timedelta(days=13))
cov_recent=float(np.mean(np.asarray(cover)[recent_test.to_numpy()]))
cov_clima=min(cov_train,cov_recent)
print("Ventanas train/validación/test:",int(train_mask.sum()),int(val_mask.sum()),int(test_mask.sum()),"| cobertura secuencia clima:",f"{cov_clima:.1%}")
if train_mask.sum()<1000 or val_mask.sum()<100 or test_mask.sum()<100: raise ValueError("Soporte diario insuficiente para entrenar/evaluar una RNN estable.")

class DeepLSTM(nn.Module):
    def __init__(self,n_in):
        super().__init__(); self.rnn=nn.LSTM(n_in,48,num_layers=2,batch_first=True,dropout=.15)
        self.head=nn.Sequential(nn.Linear(48,24),nn.ReLU(),nn.Linear(24,1))
    def forward(self,x):
        h,_=self.rnn(x); return self.head(h[:,-1,:]).squeeze(-1)

def fit_predict(use_climate):
    keep=list(range(8))+(list(range(8,14)) if use_climate else [])
    xm=X[:,:,keep]; tr=train_mask.to_numpy(); va=val_mask.to_numpy(); te=test_mask.to_numpy()
    mu=xm[tr].mean(axis=(0,1),keepdims=True); sd=xm[tr].std(axis=(0,1),keepdims=True); sd=np.where(sd<1e-6,1,sd)
    xm=np.nan_to_num((xm-mu)/sd,nan=0,posinf=0,neginf=0)
    device=torch.device("cpu"); model=DeepLSTM(xm.shape[2]).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5); loss_fn=nn.MSELoss()
    idx=np.flatnonzero(tr)
    if len(idx)>25000: idx=np.random.default_rng(42).choice(idx,25000,replace=False)
    tx=torch.tensor(xm[idx],dtype=torch.float32); ty=torch.tensor(yt[idx],dtype=torch.float32)
    vx=torch.tensor(xm[va],dtype=torch.float32); vy=torch.tensor(yt[va],dtype=torch.float32)
    best=float("inf"); best_state=None; patience=0
    for epoch in range(12):
        model.train(); perm=torch.randperm(len(tx))
        for ix in perm.split(512):
            opt.zero_grad(); loss=loss_fn(model(tx[ix]),ty[ix]); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad(): score=float(loss_fn(model(vx),vy))
        if score<best-1e-5: best=score;best_state={k:v.detach().clone() for k,v in model.state_dict().items()};patience=0
        else: patience+=1
        if patience>=3: break
    model.load_state_dict(best_state);model.eval()
    with torch.no_grad(): pp=model(torch.tensor(xm[te],dtype=torch.float32)).numpy()
    return np.maximum(0,np.expm1(pp)*sc[te]),best,epoch+1

torch.set_num_threads(4)
climate_gap_days=int((date_max-climate_map.index.max()).days)
use_climate=(cov_clima>=.80) and (climate_gap_days<=7)
meta_test=meta.loc[test_mask].copy().reset_index(drop=True)
p_clima,best_clima,epochs_clima=fit_predict(True)
p_sin,best_sin,epochs_sin=fit_predict(False)
meta_test["RNN_con_clima"]=p_clima;meta_test["RNN_sin_clima"]=p_sin
meta_test["RNN_auto"]=p_clima if use_climate else p_sin
best_val=best_clima if use_climate else best_sin;epochs=epochs_clima if use_climate else epochs_sin
meta_test["RNN_climate_mode"]="con clima" if use_climate else "sin clima: cobertura <80%"
meta_test["media7_observada"]=meta_test.media7_observada.fillna(meta_test.ultimo_valor)
rows=[]
metodos=[("RNN automática","RNN_auto")]
metodos += [("RNN con clima (diagnÃ³stico)","RNN_con_clima"),("RNN sin clima","RNN_sin_clima")]
metodos += [("Último día reportado","ultimo_valor"),("Promedio 7 días observados","media7_observada"),("Mismo día año anterior","mismo_dia_anio_anterior")]
for name,col in metodos:
    z=meta_test.loc[meta_test[col].notna()]
    rows.append({"modelo":name,"n":len(z),"MAE_tallos":(z.y-z[col]).abs().mean(),"WAPE_pct":100*(z.y-z[col]).abs().sum()/z.y.sum(),"sesgo_pct":100*(z[col]-z.y).sum()/z.y.sum()})
resultados=pd.DataFrame(rows);display(resultados)
print("Modo climático:",meta_test.RNN_climate_mode.iloc[0],"| cobertura secuencia:",f"{cov_clima:.1%}","| atraso clima (días):",climate_gap_days,"| mejor MSE validación:",best_val,"| épocas:",epochs)
meta_test.to_parquet(OUT/"predicciones_rnn_diaria.parquet",index=False);resultados.to_csv(OUT/"resultados_rnn_diaria.csv",index=False)


Último día observado: 2026-10-03 | última semana completa: 2026-09-21
La RNN usa 28 días de secuencia reciente y el valor/estado de hace 364 días.


Ventanas train/validación/test: 54095 5234 2409 | cobertura secuencia clima: 97.1%


,modelo,n,MAE_tallos,WAPE_pct,sesgo_pct
0,RNN automática,2409,567.494080,27.167414,-2.568665
1,RNN con clima (diagnÃ³stico),2409,625.251221,29.932394,-13.638431
2,RNN sin clima,2409,567.494080,27.167414,-2.568665
3,Último día reportado,2067,824.146105,38.271371,-2.873200
4,Promedio 7 días observados,2408,578.948798,27.706372,-1.324836
5,Mismo día año anterior,2187,963.058985,42.901070,-13.657882


Modo climático: sin clima: cobertura <80% | cobertura secuencia: 97.1% | atraso clima (días): 12 | mejor MSE validación: 0.040914442390203476 | épocas: 11
